# Fine-tune Surya OCR 2.5 with ms-swift

For each epoch: make new training pages (a new seed), train on them, then read a few KITAB-Bench images to see
if the model got better. At the end: the original Surya against the trained one on KITAB-Bench.

**Once:** in Colab's *Secrets* (key icon, left), add `GITHUB_TOKEN` with a read-only token for the repository.
**Run:** *Runtime > Change runtime type > T4 GPU*, then *Runtime > Run all*.

The pages are made on Colab's own disk. The model is saved to `MyDrive/surya/run_swift/model` after every
epoch: after a disconnect, *Run all* continues with the next epoch.

In [ ]:
%pip install -q "ms-swift==4.6.0" "transformers==5.19.0" "bitsandbytes==0.50.2" "qwen_vl_utils==0.0.14" decord jiwer

In [ ]:
import base64, json, os, random, re, shutil, subprocess
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["IMAGE_MIN_TOKEN_NUM"] = "49"     # Surya's smallest image (1792x28 pixels, in 32x32 pieces)
os.environ["IMAGE_MAX_TOKEN_NUM"] = "6144"   # Surya's largest image (3072x2048 pixels)

from google.colab import drive, userdata
drive.mount("/content/drive")

EPOCHS = 5                 # each epoch: new pages, made with the epoch number as seed
PAGES = 1000               # pages per epoch
CHECK = 2                  # KITAB-Bench images per dataset read after every epoch (13 datasets)
MAX_LENGTH = 6144          # longer pages are skipped; if the GPU runs out of memory, use 4096

MODEL = "datalab-to/surya-ocr-2.5"
RUN = Path("/content/drive/MyDrive/surya/run_swift")      # the trained model and the scores (Drive)
CACHE = Path("/content/drive/MyDrive/surya/cache")         # photo and text pools, kept between runs (Drive)
CODE = Path("/content/ocr")                                # our generator and its tools (Colab's disk)
PAGES_DIR = Path("/content/pages")                         # the training pages (Colab's disk)
RUN.mkdir(parents=True, exist_ok=True)

def sh(command):
    """Runs a command and stops the notebook if it fails."""
    if subprocess.run(command, shell=True).returncode:
        raise RuntimeError("This failed: " + command)

## The page generator

Our repository and the three tools it uses, at the versions it was tested with, in their own Python
environment. The photo and text pools come from Drive (made once, the first time, about half an hour).

In [ ]:
REPO = CODE / "arabic-template-bank"
if not REPO.exists():
    CODE.mkdir(exist_ok=True)
    login = base64.b64encode(f"x-access-token:{userdata.get('GITHUB_TOKEN')}".encode()).decode()
    if subprocess.run(["git", "-c", f"http.extraHeader=Authorization: Basic {login}", "clone", "-q", "--depth", "1",
                       "-b", "surya", "https://github.com/Yousef10p/OCRSynthGen.git", str(REPO)]).returncode:
        raise RuntimeError("Could not get the repository: is GITHUB_TOKEN right?")
    for name, url, commit in [("synthdoc", "https://github.com/MatterWaveAI/synthdoc.git", "1c4506dccf87984f3d8e7fdfe560fe125bd1995a"),
                              ("ocrsmith", "https://github.com/atlasia-ma/OCRSmith.git", "7ebaef82a8883499f76bd56001eab01e61ae707c"),
                              ("scan-simulator", "https://github.com/s1mb1o/scan-simulator.git", "5befedffa2c43cca599887d512785322b4b0f3c6")]:
        sh(f"git init -q {CODE / name} && cd {CODE / name} && git fetch -q --depth 1 {url} {commit} && git checkout -q FETCH_HEAD")

PY = "/content/venv/bin/python"
if not Path(PY).exists():
    sh("pip install -q uv && uv venv -q --python 3.12 /content/venv")
    sh(f"uv pip install -q --python {PY} -r {REPO}/requirements-pipeline.txt")
    sh(f"{PY} -m playwright install --with-deps chromium > /dev/null")
    site = subprocess.run([PY, "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
                          capture_output=True, text=True).stdout.strip()
    Path(site, "ocr_sources.pth").write_text("\n".join(str(p) for p in [
        CODE / "synthdoc" / "src", REPO / "synthdoc_arabic_ext", CODE / "ocrsmith" / "src", CODE / "scan-simulator"]))

CACHE.mkdir(parents=True, exist_ok=True)
for pool, build in [("text_pool", "pipeline.text_pool --articles 5000"), ("img_pool", "pipeline.img_pool")]:
    if not (REPO / pool).exists():
        if not (CACHE / f"{pool}.tar").exists():
            sh(f"cd {REPO} && {PY} -m {build} && tar -cf {CACHE / pool}.tar {pool}")
        sh(f"tar -xf {CACHE / pool}.tar -C {REPO}")
print("generator ready")

## Three settings ms-swift needs for Surya

Surya is a Qwen3.5 model with its own vocabulary, so ms-swift's Qwen3.5 defaults are corrected here.

In [ ]:
from transformers import AutoProcessor, AutoTokenizer, TokenizersBackend
from swift.template import TEMPLATE_MAPPING
from swift.template.templates.qwen import Qwen3_5Template

# 1. Surya's own tokenizer (transformers would load a Qwen3.5 tokenizer that breaks Arabic into pieces)
surya_tokenizer = TokenizersBackend.from_pretrained(MODEL)
load_processor = AutoProcessor.from_pretrained
AutoProcessor.from_pretrained = lambda *args, **kwargs: load_processor(*args, **{**kwargs, "tokenizer": surya_tokenizer})
AutoTokenizer.from_pretrained = lambda *args, **kwargs: surya_tokenizer

# 2. Surya's image token ids (ms-swift has Qwen's: 248056 and 248057)
Qwen3_5Template.image_token_id, Qwen3_5Template.video_token_id = 11, 12

# 3. the end token added by its id, so it is never merged with the answer's last characters
TEMPLATE_MAPPING["qwen3_5"].suffix = [["eos_token_id"]]

## Reading KITAB-Bench

KITAB-Bench's 13 Arabic OCR datasets. `kitab(n)` picks `n` images from each, the same ones every time;
`score()` gives the character and word error rates the way KITAB-Bench measures them.

In [ ]:
import gc, jiwer, torch
from datasets import load_dataset
from transformers import Qwen3_5ForConditionalGeneration

KITAB = ["patsocr", "historicalbooks", "khattparagraph", "synthesizear", "historyar", "adab", "muharaf",
         "onlinekhatt", "khatt", "isippt", "arabicocr", "hindawi", "evarest"]
PAGE_PROMPT = "OCR this image to HTML. Each block is a div with data-label and data-bbox (x0 y0 x1 y1, normalized 0-1000)."
BLOCK_PROMPT = "OCR this block image to HTML."
processor = AutoProcessor.from_pretrained(MODEL)
processor.image_processor.size = {"shortest_edge": 1792 * 28, "longest_edge": 3072 * 2048}


def kitab(n):
    """[(dataset, image, true text)]: n images from each dataset, the same every time."""
    picks = []
    for name in KITAB:
        ds = load_dataset(f"ahmedheakl/arocrbench_{name}", split="train")
        column = "answer" if "answer" in ds.column_names else "text"
        for i in random.Random(name).sample(range(len(ds)), min(n, len(ds))):
            if (ds[i][column] or "").strip():
                picks.append((name, ds[i]["image"].convert("RGB"), ds[i][column]))
    return picks


def read(model, image, truth):
    """The model's text for one image: Surya's page prompt for a page, its block prompt for a line or two."""
    prompt = PAGE_PROMPT if len(truth.split()) > 40 else BLOCK_PROMPT
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=4096, do_sample=False)
    return processor.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def normalize(text):
    """KITAB-Bench's normalization, after removing Surya's block labels, boxes, tags and picture descriptions."""
    text = re.sub(r'<p class="img-desc">.*?</p>|<img[^>]*>', " ", text)
    text = re.sub(r"<\|[^|>]*\|>|<[^>]+>", " ", text)
    text = re.sub(r"[\u064b-\u0670\u065f]", "", text)                     # no diacritics
    text = re.sub("[إأٱآا]", "ا", text).replace("ة", "ه").replace("ى", "ي").replace("ـ", "")
    return " ".join(text.split())


def score(truths, reads):
    """(CER, WER) in %, lower is better."""
    truths, reads = [normalize(t) for t in truths], [normalize(r) for r in reads]
    return 100 * jiwer.cer(truths, reads), 100 * jiwer.wer(truths, reads)


def check(path):
    """CER and WER of the model at path on the KITAB images read after every epoch."""
    model = Qwen3_5ForConditionalGeneration.from_pretrained(path, dtype=torch.float32).to("cuda").eval()
    reads = [read(model, image, truth) for _, image, truth in CHECK_IMAGES]
    del model
    gc.collect(); torch.cuda.empty_cache()
    return score([truth for _, _, truth in CHECK_IMAGES], reads)


CHECK_IMAGES = kitab(CHECK)
print(len(CHECK_IMAGES), "KITAB-Bench images for the check after every epoch")

## Train, one epoch at a time

Each epoch: new pages (made while the previous epoch trains), one pass over them starting from the model so far,
the model saved to Drive, then the KITAB check. Epochs already in `scores.json` are skipped.

In [ ]:
from swift import SftArguments, sft_main

def make_pages(epoch):
    """Starts making one epoch's pages in the background; returns the running process."""
    out = PAGES_DIR / f"epoch_{epoch:03d}"
    return subprocess.Popen(f"cd {REPO} && {PY} -m pipeline.dataset --n {PAGES} --seed {epoch} --out {out} "
                            f"--task full_ocr+block_ocr --tashkeel none --val 0 --workers {os.cpu_count()} "
                            f"> {PAGES_DIR}/epoch_{epoch:03d}.log 2>&1", shell=True)

def training_file(epoch):
    """The epoch's pages as ms-swift reads them: the image, Surya's prompt, our label as the answer."""
    folder = PAGES_DIR / f"epoch_{epoch:03d}"
    with open("/content/train.jsonl", "w", encoding="utf-8") as f:
        for line in (folder / "manifest.jsonl").read_text(encoding="utf-8").splitlines():
            page = json.loads(line)
            row = {"messages": [{"role": "user", "content": "<image>" + (folder / page["prompts"]).read_text(encoding="utf-8")},
                                {"role": "assistant", "content": (folder / page["targets"]).read_text(encoding="utf-8")}],
                   "images": [str(folder / page["images"])]}
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

scores_file = RUN / "scores.json"
scores = json.loads(scores_file.read_text()) if scores_file.exists() else {}
if "0" not in scores:
    scores["0"] = check(MODEL)            # the original Surya, before training
    scores_file.write_text(json.dumps(scores))

PAGES_DIR.mkdir(exist_ok=True)
todo = [e for e in range(1, EPOCHS + 1) if str(e) not in scores]
making = {todo[0]: make_pages(todo[0])} if todo else {}
for epoch in todo:
    if making.pop(epoch).wait():
        raise RuntimeError(f"Making the pages of epoch {epoch} failed: see {PAGES_DIR}/epoch_{epoch:03d}.log")
    if epoch + 1 in todo:
        making[epoch + 1] = make_pages(epoch + 1)       # the next epoch's pages, while this one trains
    training_file(epoch)

    start = str(RUN / "model") if (RUN / "model" / "config.json").exists() else MODEL
    checkpoints = sorted((RUN / "checkpoints").glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
    result = sft_main(SftArguments(
        model=start, model_type="qwen3_5", template="qwen3_5", use_hf=True,
        tuner_type="full", freeze_vit=True, freeze_aligner=False,   # vision encoder frozen, the rest trained
        torch_dtype="float32", attn_impl="sdpa",                    # a T4 has no bfloat16
        dataset=["/content/train.jsonl"], max_length=MAX_LENGTH, truncation_strategy="delete",
        add_non_thinking_prefix=False,                              # Surya's answers have no <think> part
        per_device_train_batch_size=1, gradient_accumulation_steps=8, learning_rate=1e-5,
        num_train_epochs=1, warmup_ratio=0.05, optim="adamw_bnb_8bit", gradient_checkpointing=True,
        logging_steps=10, save_steps=100, save_total_limit=1, report_to=["none"],
        output_dir=str(RUN / "checkpoints"), add_version=False,
        resume_from_checkpoint=str(checkpoints[-1]) if checkpoints else None))
    shutil.copytree(result["last_model_checkpoint"], RUN / "model", dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("optimizer*", "scheduler*", "rng_state*"))
    surya_tokenizer.save_pretrained(RUN / "model")
    shutil.rmtree(RUN / "checkpoints")                 # this epoch is done; its model is in RUN/model
    shutil.rmtree(PAGES_DIR / f"epoch_{epoch:03d}")    # its pages are not needed any more
    del result
    gc.collect(); torch.cuda.empty_cache()

    scores[str(epoch)] = check(str(RUN / "model"))
    scores_file.write_text(json.dumps(scores))
    print(f"epoch {epoch}: KITAB CER {scores[str(epoch)][0]:.1f}%  WER {scores[str(epoch)][1]:.1f}%")

print("epoch   CER    WER   (0 = the original Surya; lower is better)")
for epoch, (cer, wer) in sorted(scores.items(), key=lambda kv: int(kv[0])):
    print(f"{epoch:>5}  {cer:5.1f}  {wer:5.1f}")

## KITAB-Bench: before and after

`SAMPLES` images from each of the 13 datasets, read by the original Surya and by the trained model.

In [ ]:
SAMPLES = 10
images = kitab(SAMPLES)
reads = {}
for name, path in [("before", MODEL), ("after", str(RUN / "model"))]:
    model = Qwen3_5ForConditionalGeneration.from_pretrained(path, dtype=torch.float32).to("cuda").eval()
    reads[name] = [read(model, image, truth) for _, image, truth in images]
    del model
    gc.collect(); torch.cuda.empty_cache()

print(f"{'dataset':16s} {'CER before':>10s} {'CER after':>10s} {'WER before':>11s} {'WER after':>10s}")
for dataset in KITAB + ["ALL"]:
    rows = [k for k, (d, _, _) in enumerate(images) if dataset in (d, "ALL")]
    truths = [images[k][2] for k in rows]
    (cb, wb), (ca, wa) = [score(truths, [reads[name][k] for k in rows]) for name in ("before", "after")]
    print(f"{dataset:16s} {cb:10.1f} {ca:10.1f} {wb:11.1f} {wa:10.1f}")
print("Lower is better.")